# BirdCLEF+ 2026 — SED Inference (Self-Trained EfficientNet-B0 Ensemble)

5-fold ensemble of EfficientNet-B0 SED models (Noisy Student self-training, Stage 2).

**Pipeline**:
- Load 5× PyTorch checkpoints from dataset
- Sliding 20s window (5s stride) inference on test soundscapes
- Average ensemble predictions across folds
- Map per-window predictions to row_id → submission CSV


In [ ]:
import os
import math
import time
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import torchaudio.transforms as T
import torch
import torch.nn as nn
import timm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

NOTEBOOK_START = time.time()
BUDGET_SECS    = 85 * 60

# ── Audio / mel parameters (must match training) ──────────────────────────────
SAMPLE_RATE   = 32_000
CHUNK_SAMPLES = 20 * SAMPLE_RATE   # 640 000 samples = 20 s
WIN_STRIDE    = 5  * SAMPLE_RATE   # 5 s stride
N_MELS        = 224
N_FFT         = 4096
HOP_LENGTH    = 1252
F_MIN         = 0
F_MAX         = 16_000
TOP_DB        = 80
N_CLASSES     = 234
BACKBONE      = 'tf_efficientnet_b0.ns_jft_in1k'

_ncpu = os.cpu_count() or 4
print(f'CPU threads: {_ncpu}')
torch.set_num_threads(_ncpu)

# ── Mel transform (CPU, reused across all files) ──────────────────────────────
_mel_transform   = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
    power=2.0, norm='slaney', mel_scale='slaney', center=True,
)
_amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=TOP_DB)


def waveform_to_mel(wav: np.ndarray) -> torch.Tensor:
    """(CHUNK_SAMPLES,) → (1, 3, N_MELS, T) float32 tensor."""
    t      = torch.from_numpy(wav).float().unsqueeze(0)  # (1, L)
    mel    = _mel_transform(t)
    mel_db = _amplitude_to_db(mel)
    mel_db = mel_db - mel_db.min()
    peak   = mel_db.max()
    if peak > 0:
        mel_db = mel_db / peak
    return mel_db.repeat(3, 1, 1).unsqueeze(0)  # (1, 3, N_MELS, T)


# ── Model architecture (mirrors src/model.py) ─────────────────────────────────
class GEMFrequencyPool(nn.Module):
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=2).pow(1.0 / self.p)


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE, n_classes=N_CLASSES,
                 in_channels=3, gem_p=3.0, att_dropout=0.3):
        super().__init__()
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            self.backbone = timm.create_model(
                backbone_name, pretrained=False,
                features_only=True, out_indices=(4,), in_chans=in_channels,
            )
        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, N_MELS, 512)
            c_out  = self.backbone(dummy)[-1].shape[1]
        self.gem_pool = GEMFrequencyPool(p=gem_p)
        self.cls_conv = nn.Conv1d(c_out, n_classes, kernel_size=1)
        self.att_conv = nn.Sequential(
            nn.Conv1d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(c_out),
            nn.ReLU(inplace=True),
            nn.Dropout(att_dropout),
            nn.Conv1d(c_out, n_classes, kernel_size=1),
        )

    def forward(self, x):
        feat         = self.backbone(x)[-1]
        feat         = self.gem_pool(feat)
        frame_logits = self.cls_conv(feat).permute(0, 2, 1)
        att_weights  = torch.softmax(
            self.att_conv(feat).permute(0, 2, 1), dim=1
        )
        clip_logits  = (frame_logits * att_weights).sum(dim=1)
        return torch.sigmoid(clip_logits)  # (B, N_CLASSES)


print('Architecture defined.')


In [ ]:
# ── Locate .pt checkpoints ───────────────────────────────────────────────────
_pt_hits = subprocess.run(
    ['find', '/kaggle/input', '-name', '*.pt', '-not', '-path', '*/perch/*'],
    capture_output=True, text=True
).stdout.strip().splitlines()
_pt_hits = sorted(x for x in _pt_hits if x)
print(f'PT checkpoints found: {len(_pt_hits)}')
for p in _pt_hits:
    print(' ', p)

assert _pt_hits, 'No .pt checkpoints found under /kaggle/input'

# ── Load models ───────────────────────────────────────────────────────────────
models = []
for p in _pt_hits:
    m = BirdSEDModel()
    m.load_state_dict(torch.load(p, map_location='cpu'))
    m.eval()
    models.append(m)
    print(f'Loaded: {Path(p).name}')

# Warmup
dummy = torch.zeros(1, 3, N_MELS, 512)
with torch.no_grad():
    for m in models:
        m(dummy)
print(f'Warmup done. Ensemble size: {len(models)}')

# ── Locate competition data (search recursively) ──────────────────────────────
_sub_hits = subprocess.run(
    ['find', '/kaggle/input', '-name', 'sample_submission.csv',
     '-not', '-path', '*/stevewatson999/*'],
    capture_output=True, text=True
).stdout.strip().splitlines()
_sub_hits = [x for x in _sub_hits if x]
print(f'sample_submission.csv candidates: {_sub_hits}')
assert _sub_hits, 'Cannot find sample_submission.csv'
COMP_DATA = Path(_sub_hits[0]).parent
print(f'Competition data: {COMP_DATA}')

sample_sub   = pd.read_csv(COMP_DATA / 'sample_submission.csv')
COMP_SPECIES = list(sample_sub.columns[1:])
assert len(COMP_SPECIES) == N_CLASSES, f'Expected {N_CLASSES} species, got {len(COMP_SPECIES)}'
print(f'Species: {len(COMP_SPECIES)}')


In [ ]:
# ── Inference helpers ────────────────────────────────────────────────────────

def load_audio(path: Path) -> np.ndarray:
    data, sr = sf.read(str(path), dtype='float32', always_2d=True)
    data = data.mean(axis=1) if data.shape[1] > 1 else data[:, 0]
    if sr != SAMPLE_RATE:
        raise ValueError(f'Expected {SAMPLE_RATE} Hz, got {sr} Hz')
    mx = np.abs(data).max()
    return data / mx if mx > 0 else data


def pad_or_crop(wav: np.ndarray, length: int) -> np.ndarray:
    n = len(wav)
    if n == length:
        return wav
    if n < length:
        reps = -(-length // n)
        wav  = np.tile(wav, reps)
    return wav[:length]


@torch.no_grad()
def predict_file(waveform: np.ndarray) -> np.ndarray:
    """Sliding 20s window with 5s stride → (n_windows, N_CLASSES) float32."""
    total   = len(waveform)
    n_win   = max(1, math.ceil((total - CHUNK_SAMPLES) / WIN_STRIDE) + 1)
    results = []

    for i in range(n_win):
        start   = i * WIN_STRIDE
        seg     = pad_or_crop(waveform[start:start + CHUNK_SAMPLES], CHUNK_SAMPLES)
        mel     = waveform_to_mel(seg)  # (1, 3, N_MELS, T)

        probs_acc = torch.zeros(N_CLASSES)
        for m in models:
            probs_acc += m(mel)[0]      # (N_CLASSES,)
        results.append((probs_acc / len(models)).numpy().astype(np.float32))

    return np.stack(results, axis=0)  # (n_win, N_CLASSES)


def row_ids_for(stem: str, n_win: int) -> list:
    return [f'{stem}_{(i + 1) * 5}' for i in range(n_win)]


In [ ]:
# ── Run inference on all test soundscapes ──────────────────────────────────
UNIFORM_SCORE = 1.0 / N_CLASSES

# Try direct path first, then recursive find as fallback
test_files = sorted((COMP_DATA / 'test_soundscapes').glob('*.ogg'))
if not test_files:
    _ogg_hits = subprocess.run(
        ['find', '/kaggle/input', '-name', '*.ogg', '-path', '*/test_soundscapes/*'],
        capture_output=True, text=True
    ).stdout.strip().splitlines()
    test_files = sorted(Path(p) for p in _ogg_hits if p)
    if test_files:
        COMP_DATA = test_files[0].parent.parent
        print(f'Found test_soundscapes via recursive search: {test_files[0].parent}')
print(f'Test soundscapes: {len(test_files)}')

all_ids   = []
all_probs = []

if not test_files:
    print('No test files — generating uniform-score placeholder submission.')
    for rid in sample_sub['row_id']:
        all_ids.append(rid)
        all_probs.append(np.full((1, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))
else:
    for fpath in tqdm(test_files, desc='Inference'):
        elapsed = time.time() - NOTEBOOK_START
        if elapsed > BUDGET_SECS:
            print(f'\n⚠ Time budget reached ({elapsed/60:.1f} min) — '
                  f'filling remaining files with uniform scores')
            for fp in test_files[len(all_probs):]:
                n_win = 12
                all_ids.extend(row_ids_for(fp.stem, n_win))
                all_probs.append(np.full((n_win, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))
            break

        try:
            waveform = load_audio(fpath)
            probs    = predict_file(waveform)
            all_ids.extend(row_ids_for(fpath.stem, len(probs)))
            all_probs.append(probs)
        except Exception as e:
            print(f'  [warn] {fpath.name}: {e}')
            n_win = 12
            all_ids.extend(row_ids_for(fpath.stem, n_win))
            all_probs.append(np.full((n_win, N_CLASSES), UNIFORM_SCORE, dtype=np.float32))

probs_matrix = np.concatenate(all_probs, axis=0)
print(f'Total windows: {len(probs_matrix)}  |  elapsed: {(time.time()-NOTEBOOK_START)/60:.1f} min')


In [ ]:
# ── Build & save submission ──────────────────────────────────────────────────
sub = pd.DataFrame(probs_matrix, columns=COMP_SPECIES)
sub.insert(0, 'row_id', all_ids)

sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')
sub[COMP_SPECIES] = sub[COMP_SPECIES].fillna(UNIFORM_SCORE)

print(f'Submission shape: {sub.shape}')
print(f'Missing values:   {sub.isnull().sum().sum()}')
print(sub.head(3))

out_path = Path('/kaggle/working/submission.csv')
sub.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
